# Krea 2 Turbo style test (L4)

Same question as `flux_schnell_test.ipynb`, other direction: is the ~55-60s/image Krea2
hit on Kaggle's T4 x2 actually a T4 problem, or does it follow the model onto an L4?
Standalone -- doesn't touch `notebooks/studio.ipynb` or `scripts/server.py` (the Kaggle
pipeline, preserved on the `krea2` branch).

**This is a real port, not a rerun** -- `scripts/server.py`'s load path carries three T4-
specific workarounds this notebook deliberately drops:

1. **The fp16 patches** (`scripts/patches/t4_fp16.json`). They exist to override a
   hardcoded `dtype = torch.bfloat16` in Wan2GP's `krea2_main.py`, because Turing (the T4)
   has no native bf16. Ada (the L4) does -- that hardcoded line is already correct here, so
   this notebook loads unpatched and asks for `dtype=torch.bfloat16` throughout instead of
   `float16`. If the patch file's own `_README` is right, this should just work.
2. **The GPU1 text-tower split** (`gpu1_text_tower()` in server.py). That trick exists
   because mmgp is single-GPU-only and the T4's 16GB forced the text tower off GPU0 to fit
   the transformer. One 24GB L4 doesn't have that problem -- everything stays on one
   device, no split code needed.
3. **Aggressive host-RAM pinning tuning** (`PERC_RESERVED_MEM`). Still using mmgp's
   `offload.profile()` here (Krea 2 loads through Wan2GP's model factory, not plain
   `diffusers`, unlike the Flux test), but with roomier budgets since VRAM isn't the fight
   it was on a 16GB card.

This is a best-effort port assembled from reading `scripts/server.py`, not something I
could run myself to verify -- if `load_model()`/`offload.profile()` throw, paste the
traceback back and I'll fix the cell rather than guess twice.

In [ ]:
import os, sys, time, pathlib, subprocess

REPO = pathlib.Path.cwd()                 # assumes running from inside the pixels repo clone
WAN2GP = REPO / "Wan2GP"                   # already gitignored, same convention as studio.ipynb
WAN2GP_COMMIT = "7f06022de44538c66fa461645952d7df9d763575"  # same pin as the Kaggle notebook

def sh(c, check=True):
    print(f"$ {c}"); return subprocess.run(c, shell=True, check=check)

if not WAN2GP.exists():
    sh(f"git clone https://github.com/DeepBeepMeep/Wan2GP.git {WAN2GP}")
sh(f"cd {WAN2GP} && git checkout --quiet {WAN2GP_COMMIT}")

sh(f"pip install -q -r {WAN2GP}/requirements.txt")
sh("pip install -q mmgp hf_transfer")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# Same protobuf shim as scripts/server.py -- Wan2GP's optional deps (jax/onnxruntime/
# tensorboard) call a removed MessageFactory method at import time; harmless but noisy.
try:
    from google.protobuf import message_factory as _mf
    if not hasattr(_mf.MessageFactory, "GetPrototype"):
        if hasattr(_mf.MessageFactory, "GetMessageClass"):
            _mf.MessageFactory.GetPrototype = _mf.MessageFactory.GetMessageClass
        elif hasattr(_mf, "GetMessageClass"):
            def _get_prototype(self, descriptor):
                return _mf.GetMessageClass(descriptor)
            _mf.MessageFactory.GetPrototype = _get_prototype
except Exception:
    pass

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if torch.cuda.is_available():
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
    print("bf16 supported:", torch.cuda.is_bf16_supported())

In [ ]:
# Exact same weight set as notebooks/studio.ipynb's weights cell.
from huggingface_hub import hf_hub_download

MODELS = WAN2GP / "models"
TE = "Qwen3-VL-4B-Instruct"
TRANSFORMER = "Krea2Turbo_quanto_bf16_int8.safetensors"
FILES = [("DeepBeepMeep/krea-2", TRANSFORMER),
         ("DeepBeepMeep/krea-2", f"{TE}/{TE}_quanto_bf16_int8.safetensors"),
         ("DeepBeepMeep/krea-2", f"{TE}/config.json"),
         ("DeepBeepMeep/krea-2", f"{TE}/tokenizer.json"),
         ("DeepBeepMeep/krea-2", f"{TE}/tokenizer_config.json"),
         ("DeepBeepMeep/krea-2", f"{TE}/chat_template.jinja"),
         ("DeepBeepMeep/krea-2", f"{TE}/preprocessor_config.json"),
         ("DeepBeepMeep/Qwen_image", "qwen_vae.safetensors"),
         ("DeepBeepMeep/Qwen_image", "qwen_vae_config.json")]

MODELS.mkdir(parents=True, exist_ok=True)
missing = [(r, f) for r, f in FILES if not (MODELS / f).exists()]
print(f"{len(FILES)-len(missing)}/{len(FILES)} already on disk")
for repo, f in missing:
    hf_hub_download(repo, f, local_dir=str(MODELS))
print("weights ready")

In [ ]:
MODEL_TYPE = "krea2_turbo"  # not krea2_turbo_edit -- same choice studio.ipynb made, no vision path

sys.path.insert(0, str(WAN2GP))
os.chdir(WAN2GP)
from mmgp import offload
from shared.utils import files_locator as fl
from models.krea2.krea2_handler import family_handler

fl.set_checkpoints_paths(["models", "ckpts", "."])
tf = os.path.join("models", TRANSFORMER)
te = os.path.join("models", TE, f"{TE}_quanto_bf16_int8.safetensors")

t0 = time.time()
# dtype=bfloat16 here is Ada's *native* format, not a workaround -- no fp16 patches needed.
model, pipe = family_handler.load_model(
    model_filename=tf, model_type=MODEL_TYPE, base_model_type=MODEL_TYPE,
    model_def=family_handler.query_model_def(MODEL_TYPE, {}),
    quantizeTransformer=False, dtype=torch.bfloat16, VAE_dtype=torch.bfloat16,
    text_encoder_filename=te,
)

# No gpu1_text_tower() split -- one 24GB card holds transformer + text encoder + VAE
# together, which is the whole point of testing this here. Budgets scaled up from
# server.py's T4 numbers (11000/4000/1500) since there's no 16GB ceiling to dodge; still
# under mmgp's 80%-of-VRAM-per-budget clamp (~19.6 GiB on 24GB).
torch.cuda.set_device(0)
BUDGETS = {"transformer": 16000, "text_encoder": 6000, "vae": 2000, "*": 1000}
offload.profile(pipe, profile_no=2, quantizeTransformer=False,
                convertWeightsFloatTo=torch.bfloat16, pinnedMemory=True,
                asyncTransfers=True, budgets=BUDGETS, perc_reserved_mem_max=0.7)
offload.shared_state["_attention"] = "sdpa"
torch.cuda.empty_cache()
print(f"loaded in {round(time.time() - t0, 1)}s")

In [ ]:
# Same style/subject matrix as flux_schnell_test.ipynb, so the two results are directly
# comparable -- but at Krea2Turbo's native 8 steps, not schnell's 4.
STYLES = {
    "monoline": ("minimal hand-drawn stickman on a pure white background, thick 6px solid "
                 "black monoline ink strokes, flat saturated color fills, no gradients, no "
                 "shadows, whiteboard explainer illustration"),
    "flat_vector": ("MS Paint style drawing, thin uneven black outline, flat bucket-fill "
                    "saturated colors, no shading, no gradients, naive simple "
                    "computer-paint-program look"),
}
SUBJECTS = {
    "next_token": "a crank-operated fortune-teller machine spitting a word out of a slot-machine reel, no brain inside, just gears matching patterns",
    "hallucination": "a broken photocopier confidently printing garbage pages as if they were the original",
    "training_data": "a giant funnel pouring a torrent of documents and web pages into a grinder",
}
PROMPTS = [(s_name, sub_name, f"{s_body}. {sub_body}")
           for s_name, s_body in STYLES.items()
           for sub_name, sub_body in SUBJECTS.items()]
print(f"{len(PROMPTS)} generations queued")

In [ ]:
from PIL import Image
from IPython.display import display

OUT = REPO / "shots" / "krea2_l4_test"  # gitignored like every other shots/ subdir
OUT.mkdir(parents=True, exist_ok=True)

results = []
for style_name, subject_name, prompt in PROMPTS:
    t0 = time.time()
    try:
        out = model.generate(
            seed=7, input_prompt=prompt, n_prompt="", sampling_steps=8,
            width=1024, height=576, guide_scale=0.0, batch_size=1,
            loras_slists={"phase1": []},
        )
    except torch.cuda.OutOfMemoryError as e:
        print(f"[ OOM ] {style_name}_{subject_name} -- {e}")
        torch.cuda.empty_cache()
        continue
    secs = round(time.time() - t0, 1)

    t = out.detach().cpu() if hasattr(out, "detach") else out
    if hasattr(t, "ndim") and t.ndim == 4:      # [C, 1, H, W]
        t = t[:, 0]
    image = Image.fromarray(t.permute(1, 2, 0).numpy()) if hasattr(t, "permute") else t

    name = f"{style_name}_{subject_name}.png"
    image.save(OUT / name)
    results.append({"name": name, "style": style_name, "subject": subject_name, "s": secs})
    print(f"[{secs:>5.1f}s] {name}")
    display(image)
    torch.cuda.empty_cache()

In [ ]:
import statistics

secs = [r["s"] for r in results]
if secs:
    print(f"n={len(secs)}  mean={statistics.mean(secs):.1f}s  min={min(secs):.1f}s  max={max(secs):.1f}s")
else:
    print("no successful generations -- see the [ OOM ] lines above")
print(f"\nKaggle T4 x2 baseline: ~55-60s/image at 8 steps (power-throttled).")
print(f"FLUX.1-schnell on this L4 (discarded for quality): ~26-30s/image steady-state at 4 steps.")
print(f"Images saved to {OUT}/")